# Практическое применение декораторов

## Декораторы для измерения времени выполнения функции

### Шаг 1: Ставим задачу: как измерить, сколько времени работает функция, не изменяяя её код

<b>Представьте себе ситуацию</b>: ваш проект работает, но какая-то его часть выполняется слишком медленно. Чтобы найти "узкое место" (bottleneck), вам нужно понять, какие функции отнимают больше всего времени. Или, возможно, вы написали новый, сложный алгоритм и хотите оценить его производительность.

Задача звучит просто: <b>измерить время выполнения конкретной функции</b>.

Как бы мы подошли к ее решению, не зная о декораторах? Скорее всего, мы бы сделали что-то подобное используя стандартный модуль time:

In [1]:
import time

def process_large_data():
    # 1. Засекаем время перед началом работы
    start_time = time.time()
    
    print("Начинаю сложную обработку данных...")
    # Имитация долгой работы (например, цикл)
    _ = [i**2 for i in range(20_000_000)] 
    print("Обработка завершена.")
    
    # 2. Засекаем время после окончания
    end_time = time.time()
    
    # 3. Вычисляем и выводим разницу
    duration = end_time - start_time
    print(f"Функция 'process_large_data' выполнялась {duration:.4f} секунд.")

# Вызываем функцию, чтобы увидеть результат
process_large_data()

Начинаю сложную обработку данных...
Обработка завершена.
Функция 'process_large_data' выполнялась 2.8841 секунд.


Этот подход работает. Но у него есть несколько серьезных недостатков:

1. <b>Нарушение чистоты кода</b>: Мы <b>изменили исходный код</b> функции process_large_data. Теперь ее основная логика (обработка данных) перемешана со служебной логикой (измерение времени). Функция стала делать больше, чем должна, что нарушает принцип единой ответственности.

2. <b>Дублирование кода</b>: А что если нам нужно измерить время выполнения еще пяти или десяти других функций? Нам придется скопировать и вставить этот же код для замера времени в начало и конец <b>каждой</b> из них. Это прямое нарушение принципа <b>DRY (Don't Repeat Yourself)</b>.

3. <b>Негибкость</b>: Если мы захотим изменить формат вывода времени (например, выводить его в миллисекундах), нам придется найти и исправить это во всех фукнциях, куда мы скопировали наш код.

#### Формулируем идеальное решение

Нам нужен инструмент, который бы позволил:
- Добавить функционал замера времени к любой функции.

- Сделать это, <b>не трогая ее внутренний код</b>.

- Легко включать и отключать этот замер.

- Иметь единое место для всей логики измерения, чтобы его было легко поддерживать.

### Шаг 2: Пошагово пишем декоратор, который засекает время до и после вызова функции

Для нашей задачи нам понадобится встроенный в Python модуль time, а именно функция time.time(), которая возвращает текущее системное время в секундах (в виде числа с плавающей точкой).

Давайте вспомним наш базовый "рецепт" декоратора и будем наполнять его логикой шаг за шагом.

#### 1. Создаем "скелет" декоратора

Начнем с базовой структуры: функция timer принимает другую функцию func в качестве аргумента, определяет внутри себя обертку wrapper и возвращает эту обертку.

In [2]:
import time

def timer(func):
    def wrapper():
        # Здесь будет наша логика
        pass
    return wrapper

#### 2. Добавляем логику "ДО": засекаем время начала

Первое, что нам нужно сделать внутри wrapper - это зафиксировать время перед тем, как мы вызовем основную функцию func.

In [3]:
import time

def timer(func):
    def wrapper():
        # Засекаем время начала выполнения
        start_time = time.time()
        
        # ... здесь будет вызов func ...
        
    return wrapper

#### 3. Вызываем исходную функцию и (ВАЖНО!) обрабатываем ее результат

Теперь нам нужно вызвать саму декорируемую функцию. Но что, если эта функция что-то возвращает? Например, sum([1, 2, 3]) должно вернуть 6. Наш декоратор не должен "ломать" это поведение.

Поэтому мы вызовем func(), сохраним ее результат в переменную, а в самом конце вернем его из wrapper.

In [4]:
import time

def timer(func):
    def wrapper():
        start_time = time.time()
        
        # Вызываем исходную функцию и сохраняем то, что она вернет
        result = func()
        
        # ... здесь будет логика "ПОСЛЕ" ...
        
        # Возвращаем результат, чтобы не нарушить работу исходной функции
        return result
        
    return wrapper

Это крайне важный момент! Если wrapper ничего не будет возвращать, то любая декорированная функция, которая должна была вернуть значение, вместо этого вернет None.

#### 4. Добавляем логику "ПОСЛЕ": вычисляем и выводим время

После того как func() отработала, мы можем зафиксировать время окончания, посчитать разницу и вывести ее на экран. Для красоты вывода мы используем f-строки и форматирование числа до 4 знаков после запятой (:.4f).

Также, чтобы наш лог был информативным, мы можем получить имя декорируемой функции через ее атрибут \_\_name__.

In [5]:
import time

def timer(func):
    """
    Декоратор, который выводит время выполнения функции,
    которую он декорирует.
    """
    def wrapper():
        # 1. Засекаем время начала
        start_time = time.time()
        
        # 2. Выполняем исходную функцию
        result = func()
        
        # 3. Засекаем время окончания
        end_time = time.time()
        
        # 4. Вычисляем длительность и выводим на экран
        duration = end_time - start_time
        print(f"Функция '{func.__name__}' выполнялась {duration:.4f} секунд.")
        
        # 5. Возвращаем результат выполнения исходной функции
        return result
        
    return wrapper

Мы создаил полноценный и полезный декоратор! Он следует нашему шаблону "до-во время-после" и корректно обрабатывает возвращаемые значения.

### Шаг 3: Наглядно демонстрируем работу декоратора

Теперь у нас есть наш мощный инструмент - декоратор @timer. Давайте применим его к той самой функции process_large_data из первого шага, но на этот раз ее код останется абсолютно "чистым".

Вот полный код для демонстрации:

In [6]:
import time

# --- НАШ ГОТОВЫЙ ДЕКОРАТОР ИЗ ПРЕДЫДУЩЕГО ШАГА ---
def timer(func):
    """
    Декоратор, который выводит время выполнения функции,
    которую он декорирует.
    """
    def wrapper():
        # 1. Засекаем время начала
        start_time = time.time()
        
        # 2. Выполняем исходную функцию
        result = func()
        
        # 3. Засекаем время окончания
        end_time = time.time()
        
        # 4. Вычисляем длительность и выводим на экран
        duration = end_time - start_time
        print(f"Функция '{func.__name__}' выполнялась {duration:.4f} секунд.")
        
        # 5. Возвращаем результат выполнения исходной функции
        return result
        
    return wrapper

# --- ПРИМЕНЯЕМ ДЕКОРАТОР К НАШИМ ФУНКЦИЯМ ---

@timer
def process_large_data():
    """Имитация долгой обработки данных."""
    print("-> Начинаю сложную обработку данных...")
    # Имитация долгой работы (например, цикл)
    _ = [i**2 for i in range(20_000_000)] 
    print("-> Обработка завершена.")
    
@timer
def create_short_list():
    """Имитация быстрой операции."""
    print("-> Создаю короткий список...")
    time.sleep(0.5) # Искусственная пауза на полсекунды
    print("-> Список создан.")
    return "Готово"

# --- ВЫЗЫВАЕМ ДЕКОРИРОВАННЫЕ ФУНКЦИИ ---

print("--- Запускаем первую функцию ---")
process_large_data()

print("\n" + "="*40 + "\n")

print("--- Запускаем вторую функцию ---")
result_from_short_list = create_short_list()
print(f"Функция вернула: '{result_from_short_list}'")

--- Запускаем первую функцию ---
-> Начинаю сложную обработку данных...
-> Обработка завершена.
Функция 'process_large_data' выполнялась 3.7599 секунд.


--- Запускаем вторую функцию ---
-> Создаю короткий список...
-> Список создан.
Функция 'create_short_list' выполнялась 0.5010 секунд.
Функция вернула: 'Готово'


#### Анализ результата:

1. <b>Чистый код</b>: Посмотрите на определения функций process_large_data и create_short_list. В них нет <b>ни одной строчки кода</b>, связанной с измерением времени! Они занимаются только своей прямой задачей.

2. <b>Автоматическое измерение</b>: Просто добавив @timer над функцией, мы "включили" для нее логику измерения времени. Вывод "Функция '...' выполнялась ... секунд." был добавлен нашим декоратором автоматически.

3. <b>Переиспользование</b>: Мы использовали <b>один и тот же</b> декоратор для двух соверешнно разных функций. Нам не пришлось ничего копировать. Если мы захотим измерять время для третьей функции, мы просто добавим над ней @timer.

4. <b>Корректная работа с результатом</b>: Обратите внимание на второй вызов. Функция create_short_list вернула строку "Готово". Наш декоратор успешно "пропустил" этот результат через себя, и мы смогли сохранить его в переменную result_from_short_list и вывести на экран.

#### Итог

Мы создали наш первый практичный и переиспользуемый декоратор. Он позволит нам решить исходную задачу элегантно, не загрязняя основной код и следуя лучшим практикам программирования. Это наглядный пример того, как декораторы помогают писать более чистый, модульный и поддерживаемый код.

### Задачи

#### Задача 1: Базовый таймер

<b>Условие задачи</b>:

Напишите декоратор timer. Он должен измерить время выполнения декорируемой функции и напечатать на экран строку, содержащую это время, в формате Execution time: X.XX seconds., где X.XX — время в секундах, округленное до двух знаков после запятой.

In [7]:
from time import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time()
        result = func(*args, **kwargs)
        end = time()

        duration = end - start
        print(f'Execution time: {duration:.2f} seconds.')

        return result
    return wrapper

#### Задача 2: Таймер, сохраняющий результат

<b>Условие задачи</b>:

Напишите декоратор timer_with_return. Он должен измерить время выполнения и печатать его (в том же формате, что и в прошлой задаче). Самое важное: декорируемая функция будет <b>возвращать значение</b>. Ваш декоратор не должен "потерять" это значение.

In [8]:
import time

def timer_with_return(func):
    def wrapper(*args, **kwargs):
        start, result, end = time.time(), func(*args, *kwargs), time.time()

        duration = end - start
        print(f'Execution time: {duration:.2f} seconds.')

        return result
    return wrapper

#### Задача 3: Декоратор, возвращающий время

<b>Условие задачи</b>:

Напишите декоратор timer_returns_tuple. Этот декоратор <b>не должен ничего печатать</b>. Он должен измерить время выполнения и вернуть кортеж из двух элементов: (результат_исходной_функции, время_выполнения).

In [10]:
import time

def timer_returns_tuple(func):
    def wrapper(*args, **kwargs):
        start, result, end = time.time(), func(*args, *kwargs), time.time()
        duration = end - start

        return result, duration
    return wrapper

#### Задача 4: Точный формат вывода

<b>Условие задачи</b>:

Напишите декоратор precise_timer. Он должен измерять время и печатать его в <b>строгом</b> формате: Function 'имя_функции' took X.XXXX sec.. Имя функции должно быть получено через атрибут \_\_name__. Время должно быть отформатировано до 4 знаков после запятой.

In [12]:
import time

def precise_timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()

        duration = end - start
        print(f'Function {func.__name__!r} took {duration:.4f} sec.')

        return result
    return wrapper

#### Задача 5: Декоратор, изменяющий результат

<b>Условие задачи</b>:

Напишите декоратор add_duration_to_dict. Декорируемая функция будет возвращать словарь. Ваш декоратор должен: измерить время выполнения, добавить в возвращенный словарь новый ключ "execution_time" со значением, равным времени выполнения, и вернуть измененный словарь.

In [13]:
import time

def add_duration_to_dict(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()

        duration = end - start
        result['execution_time'] = duration

        return result
    return wrapper

## Декораторы для логирования

### Шаг 1: Объясним важность логирования в процессе разработки и отладки

На предыдущем этапе мы научились измерять производительность функций. Сегодня мы рассмотрим другую, не менее важную задачу, с которой сталкивается каждый разработчик - <b>логирование</b>.

#### Что такое логирование?

Если говорить просто, <b>логирование (logging)</b> - это процесс записи информации о событиях, происходящих во время работы вашей программы.

Наверняка каждый из вас, когда пытался найти ошибку в коде, делал что-то подобное:

In [14]:
def add_numbers(a, b):
    print(f"--- Начало работы функции add_numbers ---")
    print(f"Получены аргументы: a = {a}, b = {b}")
    result = a + b
    print(f"Функция возвращает результат: {result}")
    print(f"--- Функция add_numbers завершила работу ---")
    return result

add_numbers(10, 5)

--- Начало работы функции add_numbers ---
Получены аргументы: a = 10, b = 5
Функция возвращает результат: 15
--- Функция add_numbers завершила работу ---


15

Эти print() - и есть самая примитивная форма логирования. Они помогают нам "заглянуть" внутрь работающей программы и понять, что в ней происходит в каждый конкретный момент времени.

#### Почему print() - это плохой подход?

Хотя print() помогает при быстрой отладке, в реальных проектах он неудобен:
- <b>Засорение кода</b>: Тело функции смешивается с отладоными выводами.

- <b>Отсутствие контроля</b>: Эти сообщения нельзя легко отключить. Чтобы убраить их перед выпоуском программы, придется вручную искать и удалять все print(), что долго и рискованно.

- <b>Недостаток информации</b>: print() не добавляет никакой полезной информации, такой как время события, его важность (это просто информация, предупреждение или критическая ошибка?) или из какой части программы пришло сообщение.

#### Важность правильного логирования

Профессиональные системы логирования решают все эти проблемы и являются критически важным инструментом для любого разрботчика. Зачем они нужны?

1. <b>Отладка (Debugging)</b>: Когда программа падает с ошибкой, логи - это ваш "черный ящик". Они могут показать, какие функции вызывались перед сбоем, с какими значениями, и помочь быстро найти причину проблемы, особенно если ошибку сложно воспроизвести.

2. <b>Мониторинг (Monitoring)</b>: В работающем приложении (например, на веб-сервере) логи позволяют в реальном времени следить за его "здоровьем". Они сообщают о нормальной активности, предупреждают о потенциальных проблемах и кричат о критических сбоях.

3. <b>Анализ поведения (Analysis)</b>: Логи могут собирать статистику: какие функции используются чаще всего, какие данные обрабатываются, как пользователи взаимодействуют с системой.

#### Как нам поможет декоратор?

Задача логирования вызовов функций идеально ложится на концепцию декораторов. Мы хотим "обернуть" функцию в логику, которая будет:
- Записывать имя вызываемой функции.
- Записывать аргументы, с которыми она была вызвана.
- Записывать результат, который она вернула.

И это все - <b>не меняя ни строчки в коде самой функции</b>!

### Шаг 2: Разрабатываем декоратор, который выводит имя и аргументы функции

Итак, наша цель - создать декоратор @logger, который перед вызовом функции сообщит нам ее имя и с какими аргументами она была вызвана.

#### Стоп! А в чем сложность?

Наш предыдущий декоратор @timer работал только с функциями, у которых <b>нет аргументов</b>. Посомтрите на его внутреннюю обертку:

In [17]:
import time

def timer(func):
    def wrapper(): # <--- Нет никаких параметров!
        start = time.time()
        result = func() # <--- И здесь вызов без аргументов
        end = time.time()

        duration = end - start
        print(f'Exectuion time: {duration:.4f} seconds.')
        return result
    return wrapper

Что произойдет, если мы попытаемся применить такой декоратор к функции add(a, b)?

In [18]:
@timer # (старая версия)
def add(a, b):
    return a + b

add(10, 5) # Python попытается вызвать wrapper(10, 5)

TypeError: timer.<locals>.wrapper() takes 0 positional arguments but 2 were given

Мы получили ошибку. Python попытается передать числа 10 и 5 в нашу wrapper(), а она не умеет их принимать.

#### Решение: универсальные аргументы *args, **kwargs

Чтобы наш декоратор мог работать с <b>любой</b> функцией, независимо от количества и типа ее аргументов, нам нужно сделать нашу обертку универсальной. Для этого в Python существуют специальные конструкции:

- *args (arguments): Собирает все позиционные аругменты в кортеж (tuple).

- **kwargs (keyword arguments): Собирает все именованные аргументы в словарь (dict).

Наша новая обертка будет выглядеть так: def wrapper(*args, **kwargs):. Теперь она может "принять" абсолютно любые аргументы, не вызывая ошибки.

И, что самое важное, мы должны <b>передать</b> эти же аргументы дальше, в нашу исходную функцию, когда будем ее вызывать: func(*args, **kwargs).

#### Пишем декоратор @logger

Вооружившись этими знаниями, давайте напишем первую версию нашего логгера.

In [19]:
def logger(func):
    """
    Декоратор, который логирует имя функции и её аргументы
    перед её вызовом.
    """
    
    # 1. Создаем универсальную обертку
    def wrapper(*args, **kwargs):
        # 2. Формируем красивое сообщение для лога
        #    - func.__name__ - имя функции
        #    - args - кортеж с позиционными аргументами
        #    - kwargs - словарь с именованными аргументами
        print(f"--- Вызов функции '{func.__name__}' с аргументами: args={args}, kwargs={kwargs} ---")
        
        # 3. Вызываем исходную функцию, ПРОБРАСЫВАЯ в нее аргументы
        result = func(*args, **kwargs)
        
        # 4. Пока что просто возвращаем результат
        return result
        
    return wrapper

#### Демонстрация работы

Давайте применим наш новый декоратор к нескольким разным функциям, чтобы убедиться в его универсальности.

In [20]:
@logger
def add(a, b):
    """Простая функция сложения."""
    return a + b

@logger
def greet(name, message="Привет"):
    """Функция приветствия с именованным аргументом."""
    return f"{message}, {name}!"

# --- Вызываем и смотрим на вывод ---
print("Первый вызов:")
add(4, 8)

print("\nВторой вызов:")
greet("Алиса")

print("\nТретий вызов:")
greet(name="Боб", message="Добрый день")

Первый вызов:
--- Вызов функции 'add' с аргументами: args=(4, 8), kwargs={} ---

Второй вызов:
--- Вызов функции 'greet' с аргументами: args=('Алиса',), kwargs={} ---

Третий вызов:
--- Вызов функции 'greet' с аргументами: args=(), kwargs={'name': 'Боб', 'message': 'Добрый день'} ---


'Добрый день, Боб!'

Как видите, наш декоратор успешно "перехватил" аргументы в каждом из случаев и вывел их в консоль перед тем, как выполнить саму функцию.

#### Итог

Использование *args и **kwargs в функции-обертке - это стандартный и обязательный прием для создания по-настоящему переиспользуемых декораторов. Он позволяет нам "украшать" любую функцию, не задумываясь о ее сигнатуре (списке принимаемых аргментов).

### Шаг 3: Добавляем в декоратор логирование результата, который вернула функция

В предыдущем шаге мы написали декоратор, который сообщает нам о вызове функции. Логичным продолжением будет добавить информацию о том, чем этот вызов завершился. Нам нужно зафиксировать, какой результат вернула декорируемая функция.

Для этого нам нужно модифицировать нашу функцию wrapper. У нас уже есть строка, где мы вызываем исходную функцию и сохраняем ее результат в переменную result:

result = func(*args, **kwargs)

Все, что нам нужно сделать - это добавить print() после этой сроки, но перед return result.

#### Обновляем наш декоратор @logger

Давайте возьмем код из предыдущего шага и добавим в него всего одну строку для логирования результата.

In [21]:
def logger(func):
    """
    Декоратор, который логирует имя функции, её аргументы,
    а также возвращаемый ею результат.
    """
    
    def wrapper(*args, **kwargs):
        # --- Часть 1: Логирование "ДО" ---
        print(f"--- Вызов функции '{func.__name__}' с аргументами: args={args}, kwargs={kwargs} ---")
        
        # --- Часть 2: Выполнение самой функции ---
        result = func(*args, **kwargs)
        
        # --- Часть 3: Логирование "ПОСЛЕ" ---
        # Вот эта новая строка!
        print(f"--- Функция '{func.__name__}' вернула результат: {result} ---")
        
        # --- Часть 4: Возврат результата ---
        return result
        
    return wrapper

#### Демонстрируем работу обновленной версии

Давайте применим наш усовершенствованный декоратор к тем же функциям и посмотрим, как изменится вывод.

In [22]:
@logger
def add(a, b):
    """Простая функция сложения."""
    return a + b

@logger
def greet(name, message="Привет"):
    """Функция приветствия."""
    return f"{message}, {name}!"

@logger
def do_nothing():
    """Функция, которая ничего не возвращает (неявно вернет None)."""
    pass

# --- Вызываем и смотрим на вывод ---
print("--- Вызов add(5, 3) ---")
sum_result = add(5, 3)
print(f"  Переменная sum_result = {sum_result}\n")


print("--- Вызов greet('Ева') ---")
greet_result = greet("Ева")
print(f"  Переменная greet_result = '{greet_result}'\n")


print("--- Вызов do_nothing() ---")
none_result = do_nothing()
print(f"  Переменная none_result = {none_result}\n")

--- Вызов add(5, 3) ---
--- Вызов функции 'add' с аргументами: args=(5, 3), kwargs={} ---
--- Функция 'add' вернула результат: 8 ---
  Переменная sum_result = 8

--- Вызов greet('Ева') ---
--- Вызов функции 'greet' с аргументами: args=('Ева',), kwargs={} ---
--- Функция 'greet' вернула результат: Привет, Ева! ---
  Переменная greet_result = 'Привет, Ева!'

--- Вызов do_nothing() ---
--- Вызов функции 'do_nothing' с аргументами: args=(), kwargs={} ---
--- Функция 'do_nothing' вернула результат: None ---
  Переменная none_result = None



#### Анализ результата

1. <b>Полная картина</b>: Теперь наш лог дает нам полную "трассировку" вызова. Мы видим, с какими данными функция начала работать и с каким результатом она закончила. Это невероятно полезно при отладке: если функция возвращает не то, что вы ожидали, вы сразу увидите это в логе вместе с входными данными, которые к этому привели.

2. <b>Корректная обработка None</b>: Обратите внимание на вызов do_nothing(). Эта функция не имеет оператора return, поэтмоу по умолчанию Python возвращает из нее специальный объект None. Наш декоратор корректно перехватил и залогировал это значение.

#### Итог

Мы создали полноценный декоратор для логирования, который отслеживает полный жизненный цикл вызова функции - от получения аргументов до возврата результата. Этот инструмент можно легко "навесить" на любую функцию в вашем коде во время разработки, чтобы получить детальное представление о том, как она работает, а затем как же легко "снять", просто закомментировав строку с @logger, не трогая основной код.

### Задачи

#### Задача 1: Логгер имени функции

<b>Условие задачи</b>:

Напишите декоратор log_name. Этот декоратор должен быть универсальным и работать с любой функцией. Перед вызовом декорируемой функции он должен напечатать на экран строку в формате Calling function: 'имя_функции'.

Подсказка: даже если декорируемая функция не принимает аргументов, универсальная обертка должна содержать *args, **kwargs.

In [24]:
def log_name(func):
    def wrapper(*args, **kwargs):
        print(f'Calling function: {func.__name__!r}')
        return func(*args, **kwargs)
    return wrapper

#### Задача 2: Логгер позиционных аргументов

<b>Условие задачи</b>:

Напишите декоратор log_args. Он должен печатать кортеж с позиционными аргументами (args), с которыми была вызвана функция, в формате Args: (arg1, arg2, ...)

In [25]:
def log_args(func):
    def wrapper(*args, **kwargs):
        print(f'Args: {args}')
        return func(*args, **kwargs)
    return wrapper

#### Задача 3: Логгер именованных аргументов

<b>Условие задачи</b>:

Напишите декоратор log_kwargs. Он должен печатать словарь с именованными аргументами (kwargs), с которыми была вызвана функция, в формате Kwargs: {'key1': val1, ...}

In [26]:
def log_kwargs(func):
    def wrapper(*args, **kwargs):
        print(f'Kwargs: {kwargs}')
        return func(*args, **kwargs)
    return wrapper

#### Задача 4: Универсальный логгер аргументов

<b>Условие задачи</b>:

Напишите декоратор universal_logger. Он должен печатать и позиционные, и именованные аргументы в строгом формате: *args: (...), **kwargs: {...}.

In [27]:
def universal_logger(func):
    def wrapper(*args, **kwargs):
        print(f'*args: {args}, **kwargs: {kwargs}')
        return func(*args, **kwargs)
    return wrapper

#### Задача 5: Полный логгер

<b>Условие задачи</b>:

Напишите декоратор full_logger. Этот декоратор должен логировать полный цикл вызова функции:
1. Перед вызовом печатать строку в формате Calling '{имя_функции}' with args={...} and kwargs={...}.
2. После вызова печатать строку в формате '{имя_функции}' returned '{результат}'.

In [30]:
def full_logger(func):
    def wrapper(*args, **kwargs):
        print(f'Calling {func.__name__!r} with args={args} and kwargs={kwargs}')
        result = func(*args, **kwargs)
        print(f'{func.__name__!r} returned \'{result}\'')
        return result
    return wrapper

## Декораторы для кэширования (мемоизация)

### Шаг 1: Вводим понятие "дорогих" (ресурсоемких) вычислений

Здравствуйте! Мы уже научились использовать декораторы для логирования и измерения времени. Сегодня мы сделаем следующий шаг и научимся не просто измерять, а <b>оптимизировать</b> работу функций с помощью кэширования.

Для начала давайте разберем, что такое <b>"дорогие"</b> или <b>ресурсоемкие</b> вычисления.

<b>"Дорогая" функция</b> - это функция, выполнение которой требует значительных затрат ресурсов.

Под "ресурсами" чаще всего понимают:

- <b>Процессорное время</b>: Функция выполняет сложные математические расчеты, перебирает миллион элементов в цикле и т.д.

- <b>Время ожидания (I/O)</b>: Функция обращается к внешним системам - делает запрос в базу данных, скачивает файл из интеренета, читает большой файл с диска.

- <b>Память</b>: Функция загружает в память большие объемы данных.

#### Примеры "дорогих" функций:

##### 1. Сложные математические расчеты.

Классический пример - рекурсивное вычисление чисел Фибоначчи.

In [1]:
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

Чтобы посчитать fibonacci(5), нам нужно посчитать fibonacci(4) и fibonacci(3). Но для fibonacci(4) нам снова нужно посчитать fibonacci(3)! Получается, мы делаем одну и ту же работу <b>многократно</b>. Вызов fibonacci(40) займет уже очень заметное время.

#### 2. Запросы в сети.

Представьте функцию, которая скачивает данные о курсе валют с публичного API.

In [2]:
def get_exchange_rate(currency):
    # Делает HTTP-запрос на внешний сервер...
    # ...парсит ответ...
    # ...возвращает курс.
    pass

Каждый такой вызов - это задержка в сотни миллисекунд или даже секунд. К тому же, у многих API есть лимит на количество запросов в минуту.

#### Общая проблема: повторяющаяся работа

Ключевое наблюдение, которое объединяет эти примеры, заключается в следующем:

Для <b>одинаковых входных данных</b> эти функции <b>всегда возвращают одинаковый результат</b>.

fibonacci(5) всегда будет 5. Курс доллара к рублю, скорее всего, не изменится в течение нескольких минут. Зачем выполнять всю тяжелую работу заново, если мы уже один раз ее сделали для тех же самых входных данных?

#### Идея решения: Кэширование

Что, если бы у функции была "память" или "записная книжка"?

При первом вызове, например fibonacci(5), она бы выполнила все вычисления, получила результат 5 и <b>сохранила</b> бы его себе в "записную книжку": {"для n=5": "ответ 5"}.

При <b>повторном</b> вызове fibonacci(5) она бы не стала запускать вычисления, а просто заглянула бы в свою "память", увидела готовый ответ и мгновенно вернула бы его.

Этот процесс называется <b>кэшированием</b>, а в контексте сохранения результатов выполнения функции - <b>мемоизацией</b>.

### Шаг 2: Обясняем идею кэширования (мемоизации): сохранение результатов

В предыдущем шаге мы столкнулись с проблемой: наши "дорогие" функции выполняют одну и ту же тяжелую работу снова и снова, если мы вызываем их с одинаковыми входными данными. Идея кэширования предлагает элегантное решение этой проблемы.

Давайте представим этот процесс на простом примере из жизни.

Представьте, что вы студент, и преподаватель задаем вам сложную математическую задачу: "Чему равно 123 умножить на 456?".

<b>Первый раз</b>:
1. Вы берете ручку и бумагу.
2. Выполниете долгое вычисление в столбик (это наша "дорогая" операция).
3. Получаете ответ: 56088.
4. Вы - умный студент, поэтому вы записываете результат в свою тетрадь: (123, 456) -> 56088. Эта тетрадь - наш <b>кэш</b>.

<b>Второй раз</b>:

Через пять минут преподаватель снова спрашивает: "Чему равно 123 умножить на 456?".

Ваши действия? Вы не будеет считать все заново. Вы просто:
1. Заглянете в свою тетрадь (проверите кэш).
2. Найдете нужную запись.
3. <b>Мгновенно</b> дадите ответ: 56088.

Этот процесс и есть <b>мемоизация</b> - частный случай кэширования, который заключается в сохранении результатов выполнения функции для предотвращения повторных вычислений.

#### Переносим эту логику в программирование

Наша "тетрадь" или "записная книжка" в Python - это, как правило, словарь (dict).

- Ключом в словаре будет набор аргументов, с которыми была вызвана функция.

- Значением - результат, который функия вернула для этих аргументов.

cache = {(аргументы): результат}

Теперь мы можем описать алгоритм, по которому будет работать наша "улучшенная", кэшируемая функция:

1. <b>При вызове функции</b>: Сначала смотрим на аргументы, с которыми ее вызвали (например, n=5).

2. <b>Проверка кэша</b>: Заглядываем в наш словарь-кэш. Есть ли там уже ключ 5?

    - <b>Если ДА (Cache Hit / Попадание в кэш)</b>: Отлично! Немедленно берем сохраненное значение из словаря и возвращаем его. Основной код функции даже не запускается. Это быстрый путь.

    - <b>Если НЕТ (Cache Miss)</b>: Жаль. Такую задачу мы еще не решали:

        - Запускаем "тяжелый" оригинальный код функции и вычисляем результат.

        - <b>Важнейший шаг</b>: Перед тем как вернуть результат, <b>сохраняем его в кэш</b> для будущего использования (cache[5] = вычисленный_результат).

        - Возвращаем результат. Это <b>медленный путь</b>.

#### Почему декоратор - идеальный инструмент для этого?

Эта логика ("проверить кэш, если нет - вычислить и сохранить") является универсальной. Она не зависит от того, что именно делает кэшируемая функция - считает числа Фибоначчи или скачивает данные из интернета.

Это значит, что мы можем вынести всю эту логику в один декоратор @cache и затем применять его к любой "дорогой" функции, моментально делая ее "умнее" и быстрее!

#### Итог

Идея мемоизации заключается в том, чтобы обменять немного памяти (для хранения кэша) на значительный выигрыш в скорости. Логика проста: <b>"сначала проверь в памяти, вычисляй только в крайнем случае"</b>. Декоратор является идеальным средством для реализации этого механизма в чистом и переисользуемом виде.

### Шаг 3: Создаем простой декоратор кэширования с использованием словаря для хранения результатов

Наша задача - реализовать логику, описанную в предыдущем шаге, в виде декоратора. Нам понадобятся две ключевые вещи:

1. <b>Словарь для кэша</b>: Он будет хранить уже вычисленные результаты. Этот словарь должен создаваться один раз, когда функция декорируется, и "жить" между ее вызовами.

2. <b>Функция-обертка</b>: Она будет реализовывать логику "проверь кэш -> если нет, то вычисли и сохрани".

Давайте напишем наш декоратор.

In [3]:
def cache(func):
    """
    Простой декоратор для кэширования (мемоизации) результатов функции.
    """
    # 1. Словарь для хранения кэша. Он создается один раз
    #    и "замыкается" внутри wrapper.
    memo_cache = {}
    
    def wrapper(*args):
        # 2. Создаем ключ для словаря из аргументов.
        #    args - это кортеж, он неизменяемый (hashable),
        #    поэтому его можно использовать как ключ.
        key = args
        
        # 3. Проверяем, есть ли результат для такого ключа в кэше
        if key in memo_cache:
            # 3a. Если есть (cache hit), возвращаем его, не вызывая func
            print(f"Cache HIT для ключа {key}. Возвращаю результат из кэша.")
            return memo_cache[key]
        else:
            # 3b. Если нет (cache miss), то вычисляем результат
            print(f"Cache MISS для ключа {key}. Вычисляю результат...")
            result = func(*args)
            # Сохраняем результат в кэш перед тем, как его вернуть
            memo_cache[key] = result
            return result
            
    return wrapper

#### Разбор кода по частям:

- memo_cache = {}: Этот словарь - наше хранилище. Он определен внутри cache, но снаружи wrapper. Благодаря замыканию, wrapper всегда будет иметь доступ к <b>одному и тому же</b> объекту memo_cache при каждом своем вызове.

- def wrapper(\*args): <b>Важное ограничение!</b> В этой простой версии мы для краткости работаем только с позиционными аргументами (*args). Мы предполагаем, что все они могут быть использованы в качестве ключа словаря (т.е являются неизменяемыми, например, числа, строки, кортежи).

- key = args: Кортеж args - идеальный ключ для нашего кэша. Например, для вызова func(5, 'a'), args будет (5, 'a').

- if key in memo_cache: Та самая првоерка "записной книжки". Это самый быстрый путь.

- else: "Медленный" путь. Мы делаем всю тяжелую работу (func(*args)), но с важным дополнением - сохраняем результат в memo_cache[key] = result, чтобы в следующий раз пойти по быстрому пути.

#### Демонстрация на примере чисел Фибоначчи

Давайте примени наш декоратор к той самой "дорогой" рекурсивной функции и посмотрим на эффект.

In [4]:
@cache
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

# --- Начинаем вызовы ---
print("--- Первый вызов fibonacci(8) ---")
result = fibonacci(8)
print(f"Результат: {result}\n")

print("--- ВТОРОЙ вызов fibonacci(8) ---")
result = fibonacci(8)
print(f"Результат: {result}\n")

# --- Попробуем число побольше ---
print("--- Вызов fibonacci(10) ---")
result = fibonacci(10)
print(f"Результат: {result}\n")

--- Первый вызов fibonacci(8) ---
Cache MISS для ключа (8,). Вычисляю результат...
Cache MISS для ключа (7,). Вычисляю результат...
Cache MISS для ключа (6,). Вычисляю результат...
Cache MISS для ключа (5,). Вычисляю результат...
Cache MISS для ключа (4,). Вычисляю результат...
Cache MISS для ключа (3,). Вычисляю результат...
Cache MISS для ключа (2,). Вычисляю результат...
Cache MISS для ключа (1,). Вычисляю результат...
Cache MISS для ключа (0,). Вычисляю результат...
Cache HIT для ключа (1,). Возвращаю результат из кэша.
Cache HIT для ключа (2,). Возвращаю результат из кэша.
Cache HIT для ключа (3,). Возвращаю результат из кэша.
Cache HIT для ключа (4,). Возвращаю результат из кэша.
Cache HIT для ключа (5,). Возвращаю результат из кэша.
Cache HIT для ключа (6,). Возвращаю результат из кэша.
Результат: 21

--- ВТОРОЙ вызов fibonacci(8) ---
Cache HIT для ключа (8,). Возвращаю результат из кэша.
Результат: 21

--- Вызов fibonacci(10) ---
Cache MISS для ключа (10,). Вычисляю результат..

#### Анализ вывода

1. <b>Первый вызов fibonacci(8)</b>: Происходит много "промахов" (Cache MISS), так как кэш еще пуст. Функция рекурсивно вычисляет все значения от 0 до 8 и <b>заполняет кэш</b>.

2. <b>Второй вызов fibonacci(8)</b>: Всего один print! Происходит "попадание" (Cache HIT) для ключа (8,). Результат возвращается <b>мгновенно</b>, без каких-либо вычислений.

3. <b>Вызов fibonacci(10)</b>: Обратите внимание, что для вычисления fibonacci(10) нам нужны fibonacci(9) и fibonacci(8). fibonacci(8) уже есть в кэше, поэтому для него происходит HIT! Нам остается вычислить только fibonacci(9).

#### Итог

Мы написали простой, но полностью рабочий декоратор для кэширования. Он эффективно сохраняет результаты и значительно ускоряет повторные вызовы "дорогих" функций. Однако у него есть ограничения (например, он не работает с именованными аргументами).

### Задачи

#### Задача 1: Простой декоратор кэширования

<b>Условие задачи</b>:

Напишите декоратор cache. Он должен кэшировать результаты вызова функции. При первом вызове функции с определенными аргументами, она должна выполниться. При последующих вызовах с теми же самыми аргументами, результат должен быть возвращен из кэша, а сама функция вызываться не должна.

Подсказка: используйте словарь для хранения кэша. Кортеж args можно использовать в качестве ключа.

In [6]:
def cache(func):
    cache = {}

    def wrapper(*args):
        key = args

        if key not in cache:
            cache[key] = func(*args)

        return cache[key]
    return wrapper

#### Задача 2: Кэширование с разными аргументами

<b>Условие задачи</b>:

Напишите декоратор memoize. Он должен кэшировать результаты вызова функции для разных аргументов. Проверьте, что вызов с новым аргументом приводит к вычислению, а повторный вызов с уже использованным аргументом — к взятию из кэша.

In [7]:
def memoize(func):
    cache = {}

    def wrapper(*args):
        key = args

        if key not in cache:
            cache[key] = func(*args)

        return cache[key]
    return wrapper

#### Задача 3: Подсчет реальных вызовов

<b>Условие задачи</b>:

Напишите декоратор count_calls_cache. В ::header скрыта глобальная переменная CALL_COUNT и функция process, которая увеличивает этот счетчик при каждом своем вызове.

Ваш декоратор должен кэшировать результаты process так, чтобы при повторных вызовах с одинаковыми аргументами счетчик CALL_COUNT не увеличивался.

In [8]:
def count_calls_cache(func):
    cache = {}

    def wrapper(*args):
        key = args

        if key not in cache:
            cache[key] = func(*args)

        return cache[key]
    return wrapper

#### Задача 4: Использование lru_cache

<b>Условие задачи</b>:

Вам дана "дорогая" функция slow_operation. Импортируйте и примените к ней стандартный декоратор @lru_cache из модуля functools, чтобы оптимизировать ее работу.

In [9]:
from functools import lru_cache

#### Задача 5: Логгер попаданий в кэш

<b>Условие задачи</b>:

Напишите декоратор caching_logger. Он должен не только кэшировать результаты, но и сообщать о своих действиях.
- Если результат вычисляется впервые (промах кэша), он должен напечатать "Cache miss for key: (...)".
- Если результат берется из кэша (попадание в кэш), он должен напечатать "Cache hit for key: (...)".

Ключом является кортеж args.

In [11]:
def caching_logger(func):
    cache = {}

    def wrapper(*args):
        key = args

        if key not in cache:
            print(f'Cache miss for key: {key}')
            cache[key] = func(*args)
        else:
            print(f'Cache hit for key: {args}')

        return cache[key]
    return wrapper

## Декораторы для проверки аргументов и валидации

### Шаг 1: Обсуждаем сценарии, когда необходимо проверять входные данные функции перед её выполнением

Здравствуйте! До сих пор мы писали функции, неявно предполагая, что они всегда будут получать на вход корректные данные. Но в реальных программах, которые взаимодействуют с пользователями, файлами или внешними системами, это предположение почти никогда не выполняется.

#### Что произойдет, если в функцию придут "плохие" данные?

Давайте рассмотрим простую функцию деления:

In [12]:
def divide(a, b):
    return a / b

Что может пойти не так?
- divide(10, '2') -> TypeError: unsupported operand type(s) for /: 'int' and 'str'. Программа упадет, потому что мы не можем делить на строку.
- divide(10, 0) -> ZeroDivisionError: division by zero. Программа упадет из-за деления на ноль.

Такие ошибки могут привести к сбою всей программы. Чтобы этого избежать, программисты используют так называемое <b>защитное программирование (defensive programming)</b>, ключевой частью которого является <b>валидация входных данных</b>.

#### Традиционный подход: проверки внутри функции

Обычно, чтобы сделать функцию надежной, мы добавляем проверки в самое ее начало:

In [13]:
def divide_safe(a, b):
    # --- Блок валидации ---
    if not isinstance(a, (int, float)):
        raise TypeError(f"Аргумент 'a' должен быть числом, а не {type(a).__name__}")
    if not isinstance(b, (int, float)):
        raise TypeError(f"Аргумент 'b' должен быть числом, а не {type(b).__name__}")
    if b == 0:
        raise ValueError("Делитель 'b' не может быть равен нулю")
    # --- Конец блока валидации ---
    
    # --- Основная бизнес-логика ---
    return a / b

Этот подход работает, но у него есть те же проблемы, что мы уже видели ранее:

1. <b>Засорение кода</b>: Основная логика (return a / b) теряется среди многочисленных проверок. Тело функции раздувается и становится сложнее для чтения.

2. <b>Дублирование кода (нарушение DRY)</b>: Представьте, что у вас есть 10 разных математических функций, и в каждой из них нужно проверять, что все аргументы - это числа. Вам придется скопировать и вставить первые две проверки в каждую из этих 10 функций.

#### Когда еще нужна валидация? Типичные сценарии:

- <b>Проверка типов</b>: Убедиться, что аргумент яляется строкой, числом, списком и т.д.

- <b>Проверка диапазона значений</b>: Убедиться, что возраст находится в диапазоне от 0 до 120, или что количество товара - положительное число.

- <b>Проверка формата</b>: Проверить, что строка является корректным email-адресом или номером телефона.

- <b>Проверка на пустоту</b>: Убедиться, что переданный список или строка не пустые.

- <b>Проверка бизнес-правил</b>: Проверить, что имя пользователя уникально, или что пароль соответствует требованиям сложности.

#### Идеальное решение - декораторы

Все эти проверки - это "сквозная" логика, которую идеально вынести за пределы основной функции. Мы хотим, чтобы наша функция занималась только своим делом, а вся валидация была описана где-то в другом месте.

Мы можем создавать декораторы, которые будут декларировать требования к аргументам. Представьте, как число мог бы выглядеть наш код:

In [14]:
# @validate_types(a=(int, float), b=(int, float))
# @ensure_not_zero('b')
# def divide(a, b):
#     return a / b

Такой код гораздо легче читать. Мы сразу видим, каким требованиям должны удовлетворять аргументы.

### Шаг 2: Пишем декоратор, который проверяет типы переданных аргументов

Наша цель - создать декоратор @validate_types, который можно будет настраивать, указывая ожидаемые типы для конкретных аргументов. Например, вот так:

@validate_types(quantity=int, price=(int, float))

#### Проблема: как передать аргументы в сам декоратор?

До сих пор наши декораторы просто принимали функцию и возвращали обертку. Теперь же сам декоратор должен принять конфигурацию (ожидаемые типы). Это требует создания "фабрики декораторов" - трехуровневой структуры:

1. <b>Фабрика (validate_types)</b>: Внешняя функция, которая принимает аргументы для настройки (например, quantity=int). Ее задача - вернуть сам декоратор.

2. <b>Декоратор (decorator)</b>: Средняя функция. Она делает то, что мы уже привыкли: принимает декорируемую функцию (func) и возвращает обертку.

3. <b>Обертка (wrapper)</b>: Внутренняя функция. Она принимает аргументы вызова (*args, **kwargs), выполняет проверку типов и, если все в порядке, вызывает исходную функцию func.

#### Задача: как сопоставить *args и **kwargs с именами аргументов?

Внутри wrapper у нас есть *args (кортеж) и **kwargs (словарь). Но для проверки нам нужно знать, какое значение соответствует какому имени аргумента. Например, при вызове func(5, price=10.5) нам нужно понять, что quantity равно 5.

Для надежного решения этой задачи мы воспользуемся встроенным модулем inspect. Он позволяет "заглянуть" в структуру функции и получить информацию о ее аргументах.

#### Пишем наш декоратор @validate_types

In [15]:
import inspect

def validate_types(**expected_types):
    """
    Фабрика декораторов. Создает декоратор, который проверяет,
    что аргументы функции соответствуют ожидаемым типам.
    """
    
    # 2. Сам декоратор
    def decorator(func):
        
        # 3. Обертка, которая будет выполнять проверку
        def wrapper(*args, **kwargs):
            
            # Используем inspect, чтобы связать *args и **kwargs с именами
            bound_args = inspect.signature(func).bind(*args, **kwargs)
            bound_args.apply_defaults() # Применяем значения по умолчанию
            
            # 4. Проходим по ожидаемым типам, которые передали в фабрику
            for arg_name, expected_type in expected_types.items():
                
                # Проверяем, есть ли такой аргумент в вызове
                if arg_name in bound_args.arguments:
                    actual_value = bound_args.arguments[arg_name]
                    
                    # 5. Проверяем тип с помощью isinstance
                    if not isinstance(actual_value, expected_type):
                        raise TypeError(
                            f"Аргумент '{arg_name}' для функции '{func.__name__}' "
                            f"ожидает тип {expected_type}, но получил {type(actual_value).__name__}."
                        )
                        
            # 6. Если все проверки пройдены, вызываем исходную функцию
            return func(*args, **kwargs)
            
        return wrapper
    return decorator

#### Демонстрация работы

Давайте применим наш новый мощный декоратор к функции, которая вычисляет стоимость заказа.

In [19]:
@validate_types(quantity=int, price_per_item=(int, float))
def calculate_order_total(quantity, price_per_item):
    """Вычисляет общую стоимость заказа."""
    if quantity <= 0:
        raise ValueError("Количество должно быть положительным числом.")
    return quantity * price_per_item

# --- Тестируем ---

# 1. Корректный вызов
try:
    total = calculate_order_total(5, 10.5)
    print(f"Успех! Общая стоимость: {total}")
except (TypeError, ValueError) as e:
    print(f"Ошибка: {e}")

# 2. Некорректный тип для quantity (строка вместо int)
try:
    total = calculate_order_total("5", 10.5)
    print(f"Успех! Общая стоимость: {total}")
except (TypeError, ValueError) as e:
    print(f"Ошибка: {e}")
    
# 3. Некорректный тип для price_per_item (строка вместо int или float)
try:
    total = calculate_order_total(5, "10.5")
    print(f"Успех! Общая стоимость: {total}")
except (TypeError, ValueError) as e:
    print(f"Ошибка: {e}")

Успех! Общая стоимость: 52.5
Ошибка: Аргумент 'quantity' для функции 'calculate_order_total' ожидает тип <class 'int'>, но получил str.
Ошибка: Аргумент 'price_per_item' для функции 'calculate_order_total' ожидает тип (<class 'int'>, <class 'float'>), но получил str.


#### Анализ результата

Наш декоратор успешно перехватил вызовы с некорректными типами данных и сгенерировал понятные, информативные исключения TypeError, не позволив "плохим" данным попасть в основную логику функции. При этом код самой функции calculate_order_total остался чистым и сфоксированным на своей задаче.

#### Итог

Декораторы с аргументами ("фабрики декораторов") - это мощный инструмент для создания настраиваемых оберток. Используя их вместе с модулем inspect, мы можем писать надежные декораторы для валидации, которые отделяют логику проверки данных от основной бизнес-логики, делая код чище и безопаснее.

### Шаг 3: Рассматриваем, как выбрасывать исключения в случае неудачной валидации

Когда проверка данных внутри нашего декоратора проваливается, у нас есть несколько вариантов действий. Мы могли бы, например, просто вывести сообщение в консоль с помощью print(). Но это очень плохая практика.

Почему? Потому что print() <b>не останавливает выполнение</b> некорректного кода. Сообщение появится на экране, но после этого программа попытается выполнить основную логику функции с "плохими" данными, что, скорее всего, приведет к другой, менее понятной ошибке где-то глубже в коде.

Правильный, "питонический" способ сообщить об ошибке - это <b>сгенерировать (выбросить) исключение</b>.

#### Что такое исключение?

Исключение - это сигнал о том, что в программе произошла ошибка. Когда исключение "выбрасывается" с помощью ключевого слова raise, нормальное выполнение кода немедленно прекращается, и Python начинает искать обработчик этого исключения (try...except). Если обработчик не найден, программа аварийно завершает свою работу, выводя информацию об ошибке.

Это именно то, что нам нужно: <b>немедленно остановить операцию, если входные данные неверны</b>.

#### Выбор правильного типа исключения

В Python встроено множество исключений для разных ситуаций. Для задач валидации чаще всего используются два из них:

##### 1. TypeError

Следует выбрасывать, когда тип переданного аргумента не соответствует ожидаемому.
- <b>Пример</b>: Функция ожидает число (int), а ей передали строку (str).
- Именно это исключение мы использовали в нашем декораторе @validate_types. Посмотрите на эту строку:

In [20]:
# if not isinstance(actual_value, expected_type):
#     raise TypeError(f"Аргумент '{arg_name}' ... ожидает тип {expected_type}...")

##### 2. ValueError

Следует выбрасывать, когда <b>тип</b> аргумента правильный, но его <b>значение</b> недопустимо в данном контексте.
- <b>Пример</b>: Функция для расчета квадратного корня ожидает число, ей передали -5. Тип (int) верный, но значение - нет.
- <b>Пример</b>: Функция деления ожидает делитель-число, ей передали 0. Тип (int) верный, но значение недопустимо.

#### Создание информативных сообщений об ошибках

Просто выбросить исключение raise ValueError - это уже хорошо, но еще лучше - снабдить его понятным сообщением. Хорошее сообщение об ошибке должно отвечать на вопросы:
- <b>Что</b> пошло не так?
- <b>Где</b> это произошло (в какой функции/аргументе)?
- <b>Почему</b> (какое значение было ожидаемым, а какое получено)?

#### Пример декоратора, использующего ValueError

Давайте напишем простой декоратор, который проверяет, что числовой аргумент является положительным.

In [21]:
def ensure_positive(arg_name):
    """Фабрика декораторов: проверяет, что аргумент arg_name > 0."""
    def decorator(func):
        def wrapper(*args, **kwargs):
            # (Здесь для простоты используем inspect или просто kwargs)
            value = kwargs.get(arg_name)
            
            if value is not None and value <= 0:
                # Тип верный (число), но значение - нет. Выбрасываем ValueError!
                raise ValueError(
                    f"В функции '{func.__name__}', аргумент '{arg_name}' должен быть положительным, "
                    f"но получено значение: {value}."
                )
            return func(*args, **kwargs)
        return wrapper
    return decorator

# Применяем
@ensure_positive(arg_name='amount')
def process_payment(customer_id, amount):
    print(f"Обработка платежа на сумму {amount} для клиента {customer_id}.")

# Тестируем
try:
    process_payment(customer_id=101, amount=500)
    process_payment(customer_id=102, amount=-50) # Эта строка вызовет ошибку
except ValueError as e:
    print(f"Произошла ошибка валидации: {e}")

Обработка платежа на сумму 500 для клиента 101.
Произошла ошибка валидации: В функции 'process_payment', аргумент 'amount' должен быть положительным, но получено значение: -50.


#### Итог

Выбрасывание исключений (raise) - это стандартный и самый надежный способ обработки ошибок валидации в Python. Используйте TypeError для неверных типов и ValueError для недопустимых значений. Всегда снабжайте исключения подробными и понятными сообщениями - это значительно упростит отладку и поддержку вашего кода.

### Задачи

#### Задача 1: Валидатор типа

<b>Условие задачи</b>:

Напишите декоратор validate_string_argument. Декоратор должен проверять, что <b>первый</b> позиционный аргумент, переданный в функцию, является строкой (str).
- Если это строка, функция должна выполниться.
- Если это не строка, декоратор должен выбросить исключение TypeError с сообщением "Argument must be a string".

In [22]:
def validate_string_argument(func):
    def wrapper(*args, **kwargs):
        if not isinstance(args[0], str):
            raise TypeError('Argument must be a string')
        return func(*args, **kwargs)
    return wrapper

#### Задача 2: Валидатор положительного числа

<b>Условие задачи</b>:

Напишите декоратор ensure_positive_number. Он должен проверять, что <b>первый</b> позиционный аргумент является положительным числом (больше нуля).
- Если аргумент — положительное число, функция выполняется.
- В противном случае (если это не число, или число равно 0, или отрицательное) декоратор должен выбросить исключение ValueError с сообщением "Argument must be a positive number".

In [ ]:
def ensure_positive_number(func):
    def wrapper(*args, **kwargs):
        value = args[0]

        if not isinstance(value, int) or value <= 0:
            raise ValueError('Argument must be a positive number')
        
        return func(*args, **kwargs)
    return wrapper

#### Задача 3: Валидатор для именованного аргумента

<b>Условие задачи</b>:

Напишите декоратор validate_user_role. Он должен проверять, что в декорируемую функцию был передан именованный аргумент role со значением "admin".
- Если role равен "admin", функция выполняется.
- Если role имеет другое значение или не передан вовсе, декоратор должен выбросить исключение PermissionError с сообщением "Access denied".

In [ ]:
def validate_user_role(func):
    def wrapper(*args, **kwargs):
        role = kwargs.get('role')

        if role != 'admin':
            raise PermissionError('Access denied')

        return func(*args, **kwargs)
    return wrapper

#### Задача 4: Валидатор количества аргументов

<b>Условие задачи</b>:

Напишите декоратор check_arg_count. Он должен проверять, что в декорируемую функцию передано <b>ровно 2</b> позиционных аргумента.
- Если аргументов два, функция выполняется.
- Если их больше или меньше, декоратор должен выбросить исключение ValueError с сообщением "Function requires exactly 2 arguments".

In [25]:
def check_arg_count(count: int=2):
    def decorator(func):
        def wrapper(*args, **kwargs):
            if len(args) != count:
                raise ValueError(f'Function requires exactly {count} arguments')

            return func(*args, **kwargs)
        return wrapper
    return decorator

#### Задача 5: Декоратор "Запрет пустых строк"

<b>Условие задачи</b>:

Напишите декоратор no_empty_strings. Он должен проверить <b>все</b> позиционные аргументы, переданные в функцию. Если хотя бы один из них является пустой строкой (""), декоратор должен выбросить ValueError с сообщением "Empty strings are not allowed".

In [26]:
def no_empty_strings(func):
    def wrapper(*args, **kwargs):
        all_args = args + tuple(kwargs.values())

        if '' in all_args:
            raise ValueError('Empty strings are not allowed')

        return func(*args, **kwargs)
    return wrapper